# NB8 — Gateway / Edge Latency and Memory Profiling

Measure the computational cost of the existing attribution pipeline without retraining, changing thresholds, or using test labels to make decisions. Run on the intended gateway-class device (e.g., Raspberry Pi-class hardware) for deployment-relevant results. Desktop results are desktop measurements, not gateway evidence.

This notebook separates fitted-model inference latency from feature-table file I/O. It will not treat saved predictions as inference and will report a blocker if no fitted model is available.

## 1. Setup and paths

In [3]:
from pathlib import Path
import os, sys, json, time, platform
import numpy as np
import pandas as pd
try:
    import joblib
except ImportError:
    joblib = None
try:
    import psutil
except ImportError:
    psutil = None
HERE = Path.cwd().resolve()
roots = [p for p in [HERE, *HERE.parents] if (p / "src" / "intel_utils.py").exists()]
if not roots:
    raise FileNotFoundError("Start Jupyter inside AIoT_Research or a subfolder containing src/intel_utils.py.")
ROOT = roots[0]
PROC = ROOT / "data" / "processed"
OUT = ROOT / "outputs" / "notebook08"
OUT.mkdir(parents=True, exist_ok=True)
FEATURE_PATH = PROC / "feature_dataset_AF_v2.parquet"
PRED_PATH = ROOT / "outputs" / "notebook06_v2" / "nb6_v2_predictions.parquet"
CFG_PATH = ROOT / "outputs" / "notebook06_v2" / "nb6_v2_config.json"
print("Root:", ROOT, "| Python:", sys.version.split()[0])
print("Platform:", platform.platform(), "| Processor:", platform.processor() or "not reported")
print("Logical CPUs:", os.cpu_count())

def benchmark(fn, warmup=3, repeats=30):
    """Return wall-clock latency statistics in milliseconds."""
    for _ in range(max(0, int(warmup))):
        fn()
    samples = []
    for _ in range(max(1, int(repeats))):
        t0 = time.perf_counter()
        fn()
        samples.append((time.perf_counter() - t0) * 1000.0)
    a = np.asarray(samples, dtype=float)
    return {
        "repeats": int(len(a)),
        "median_ms": float(np.median(a)),
        "p95_ms": float(np.percentile(a, 95)),
        "p99_ms": float(np.percentile(a, 99)),
        "mean_ms": float(np.mean(a)),
    }

def rss_mb():
    """Whole-process RSS in MiB, or NaN if psutil is unavailable."""
    if psutil is None:
        return float("nan")
    try:
        return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
    except Exception:
        return float("nan")

def predict_for_profile(estimator, frame):
    """Profile raw inference while matching the estimator's fit-time input format.

    NB6's saved RandomForestClassifier was fitted without pandas feature names.
    The DataFrame is already ordered using the saved feature list; convert it to
    NumPy only for estimators that were fitted without feature names. This avoids
    sklearn's feature-name mismatch warning without suppressing warnings globally.
    Calibration/gating costs are measured separately.
    """
    model_input = frame
    if isinstance(frame, pd.DataFrame) and not hasattr(estimator, "feature_names_in_"):
        model_input = frame.to_numpy(dtype=float)

    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(model_input)
    if hasattr(estimator, "decision_function"):
        return estimator.decision_function(model_input)
    raise TypeError(f"Unsupported estimator: {type(estimator).__name__}")
rss_before = rss_mb()


Root: C:\AIoT_Research | Python: 3.12.7
Platform: Windows-10-10.0.19045-SP0 | Processor: Intel64 Family 6 Model 142 Stepping 9, GenuineIntel
Logical CPUs: 4


## 2. Discover model artifacts and inspect project outputs

In [5]:
# 2. Discover model artifacts and inspect project outputs

import json
import time
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

model_paths = []

for folder in (ROOT / "outputs", ROOT / "models"):
    if folder.exists():
        for pattern in ("*.joblib", "*.pkl", "*.pickle"):
            model_paths.extend(folder.rglob(pattern))

model_paths = sorted(
    set(p.resolve() for p in model_paths if p.is_file())
)

print("Feature table:", FEATURE_PATH.exists(), FEATURE_PATH)
print("NB6 predictions:", PRED_PATH.exists(), PRED_PATH)
print("NB6 config:", CFG_PATH.exists(), CFG_PATH)
print("Serialized model candidates:")

for p in model_paths:
    print(" -", p.relative_to(ROOT))

if CFG_PATH.exists():
    try:
        print(
            "NB6 config keys:",
            sorted(json.loads(CFG_PATH.read_text(encoding="utf-8")))
        )
    except Exception as e:
        print("Could not parse NB6 config:", repr(e))


# Load the feature table.
features = (
    pd.read_parquet(FEATURE_PATH)
    if FEATURE_PATH.exists()
    else None
)

if features is not None:
    print("Feature table shape:", features.shape)
    print("Columns:", features.columns.tolist())


# NB6 exports a dictionary containing the estimator and its metadata.
# Select a compatible artifact, preferring W/F for primary profiling.
model = None
model_artifact = None
model_path = None
model_load_seconds = None
model_features = None
imputation_medians = None
model_type = None
X = None
artifact_metadata = None

preferred_names = [
    "nb6_W_F.joblib",
    "nb6_W_G.joblib",
    "nb6_W_I.joblib",
    "nb6_ALL_F.joblib",
    "nb6_ALL_G.joblib",
    "nb6_ALL_I.joblib",
]

path_by_name = {p.name: p for p in model_paths}
ordered_paths = [
    path_by_name[name]
    for name in preferred_names
    if name in path_by_name
]
ordered_paths.extend(
    p for p in model_paths if p not in ordered_paths
)

for candidate in ordered_paths:
    t0 = time.perf_counter()

    try:
        obj = joblib.load(candidate)
        elapsed = time.perf_counter() - t0

        # Support the NB6 dictionary artifact format.
        if isinstance(obj, dict):
            estimator = obj.get("model")
            saved_features = obj.get("features")
            saved_medians = obj.get("imputation_medians")

            if estimator is None:
                print(
                    "Skipped", candidate.name,
                    "- artifact dictionary has no 'model' entry."
                )
                continue

            if not (
                hasattr(estimator, "predict_proba")
                or hasattr(estimator, "decision_function")
            ):
                print(
                    "Skipped", candidate.name,
                    "- saved model has no supported inference method."
                )
                continue

            if not saved_features:
                print(
                    "Skipped", candidate.name,
                    "- artifact dictionary has no saved 'features' list."
                )
                continue

            if not isinstance(saved_medians, dict):
                print(
                    "Skipped", candidate.name,
                    "- artifact dictionary has no valid imputation medians."
                )
                continue

            candidate_features = list(saved_features)
            candidate_medians = pd.Series(saved_medians)

        # Also support bare estimators from older exports, if present.
        else:
            estimator = obj

            if not (
                hasattr(estimator, "predict_proba")
                or hasattr(estimator, "decision_function")
            ):
                print(
                    "Skipped", candidate.name,
                    "- object is not a compatible estimator."
                )
                continue

            if not hasattr(estimator, "feature_names_in_"):
                print(
                    "Skipped", candidate.name,
                    "- bare estimator has no feature_names_in_; "
                    "cannot safely infer its input schema."
                )
                continue

            candidate_features = list(estimator.feature_names_in_)
            candidate_medians = None

        if features is None:
            print(
                "Skipped", candidate.name,
                "- feature dataset is unavailable."
            )
            continue

        missing = [
            c for c in candidate_features
            if c not in features.columns
        ]

        if missing:
            print(
                "Skipped", candidate.name,
                "- missing feature columns:", missing
            )
            continue

        if candidate_medians is not None:
            missing_medians = [
                c for c in candidate_features
                if c not in candidate_medians.index
            ]

            if missing_medians:
                print(
                    "Skipped", candidate.name,
                    "- missing saved medians:", missing_medians
                )
                continue

            candidate_medians = candidate_medians.reindex(
                candidate_features
            )

            if candidate_medians.isna().any():
                print(
                    "Skipped", candidate.name,
                    "- one or more saved medians are NaN."
                )
                continue

            candidate_X = features[candidate_features].copy()
            candidate_X = candidate_X.replace(
                [np.inf, -np.inf], np.nan
            )
            candidate_X = candidate_X.fillna(candidate_medians)

        else:
            # No imputation metadata exists for a bare estimator.
            candidate_X = features[candidate_features].copy()

            if candidate_X.isna().any().any():
                print(
                    "Skipped", candidate.name,
                    "- bare estimator input contains missing values "
                    "and no saved imputation medians are available."
                )
                continue

        if candidate_X.empty:
            print(
                "Skipped", candidate.name,
                "- prepared feature table is empty."
            )
            continue

        # Accept this artifact only after schema and preprocessing checks.
        model = estimator
        model_artifact = obj
        model_path = candidate
        model_load_seconds = elapsed
        model_features = candidate_features
        imputation_medians = candidate_medians
        model_type = type(estimator).__name__
        X = candidate_X
        artifact_metadata = (
            obj if isinstance(obj, dict) else {}
        )

        break

    except Exception as e:
        print(
            "Skipped unreadable/incompatible artifact:",
            candidate.name,
            "|",
            type(e).__name__,
            "|",
            str(e)
        )


if model is not None:
    print("\nSelected model:", model_path.relative_to(ROOT))
    print("Model type:", model_type)
    print("Model load seconds:", round(model_load_seconds, 6))
    print("Input shape:", X.shape)
    print("Model feature count:", len(model_features))
    print("Missing input cells after preprocessing:", int(X.isna().sum().sum()))
    print("Artifact regime:", artifact_metadata.get("regime"))
    print("Artifact feature set:", artifact_metadata.get("feature_set"))
    print("Saved calibration method:", artifact_metadata.get("calibration_method"))
    print("Saved tau:", artifact_metadata.get("tau"))
    print("Saved t_corr:", artifact_metadata.get("t_corr"))

else:
    print(
        "\nINFERENCE PROFILING BLOCKED: no compatible estimator "
        "with a verified feature schema and preprocessing metadata "
        "could be loaded."
    )


Feature table: True C:\AIoT_Research\data\processed\feature_dataset_AF_v2.parquet
NB6 predictions: True C:\AIoT_Research\outputs\notebook06_v2\nb6_v2_predictions.parquet
NB6 config: True C:\AIoT_Research\outputs\notebook06_v2\nb6_v2_config.json
Serialized model candidates:
 - outputs\notebook06_v2\models\nb6_ALL_F.joblib
 - outputs\notebook06_v2\models\nb6_ALL_G.joblib
 - outputs\notebook06_v2\models\nb6_ALL_I.joblib
 - outputs\notebook06_v2\models\nb6_W_F.joblib
 - outputs\notebook06_v2\models\nb6_W_G.joblib
 - outputs\notebook06_v2\models\nb6_W_I.joblib
NB6 config keys: ['calibration_methods', 'chosen', 'conditions', 'decision_policy', 'max_false_correction_rate', 'min_accepted', 'prob_floor', 'quick_test', 'regimes', 'rf_params', 'sets', 'sklearn', 'target_risk']
Feature table shape: (11826, 53)
Columns: ['window_id', 'episode_id', 'split', 'scenario', 'variant', 'event_direction', 'drift_type', 'drift_event_relation', 'interval_phase', 'label', 'label_purity', 'long_label_purity', 

## 3. Load feature data and choose a compatible fitted estimator

In [7]:
# Preserve the artifact selected and schema-checked in Section 2.
# Do not reset model/model_path here: that previously discarded the compatible
# NB6 dictionary artifact and caused profiling to report a false negative.
if features is None:
    raise FileNotFoundError(f"Feature dataset is missing: {FEATURE_PATH}")
if model is None or X is None:
    raise RuntimeError(
        "No compatible NB6 estimator was selected in Section 2. "
        "Check the artifact discovery diagnostics above; do not infer a schema."
    )
if not model_features or any(c not in features.columns for c in model_features):
    raise ValueError("Selected artifact feature schema no longer matches the feature table.")
X = X.loc[:, model_features].copy()
print("Profiling input retained:", X.shape, "| artifact:", model_path.relative_to(ROOT))


Profiling input retained: (11826, 15) | artifact: outputs\notebook06_v2\models\nb6_W_F.joblib


## 4. Benchmark model inference

In [9]:

# 4. Benchmark model inference

latency_rows = []
inference_profiled = False
input_rows_available = 0

if model is not None and X is not None and len(X) > 0:
    # Confirm the selected model has a supported inference method.
    if not (
        hasattr(model, "predict_proba")
        or hasattr(model, "decision_function")
    ):
        raise TypeError(
            f"Selected model {type(model).__name__} has no supported "
            "predict_proba or decision_function method."
        )

    # Keep exactly the saved training feature order.
    if model_features is not None:
        missing_features = [
            c for c in model_features if c not in X.columns
        ]
        if missing_features:
            raise ValueError(
                f"Profiling input is missing model features: {missing_features}"
            )
        X = X[model_features].copy()

    input_rows_available = len(X)

    # Use a small, representative single-window input.
    single_X = X.iloc[[0]].copy()
    single = benchmark(
        lambda: predict_for_profile(model, single_X),
        warmup=10,
        repeats=200
    )

    # Benchmark a batch, capped at 256 rows.
    n_batch = min(256, len(X))
    batch_X = X.iloc[:n_batch].copy()
    batch = benchmark(
        lambda: predict_for_profile(model, batch_X),
        warmup=5,
        repeats=50
    )

    latency_rows.extend([
        {
            "measurement": "single_window_inference",
            "rows_per_call": 1,
            **single
        },
        {
            "measurement": "batch_inference",
            "rows_per_call": n_batch,
            **batch
        }
    ])

    inference_profiled = True

    print("Selected artifact:", model_path.relative_to(ROOT))
    print("Model type:", type(model).__name__)
    print("Model load seconds:", round(model_load_seconds, 6))
    print("Input rows available:", input_rows_available)
    print("Features per row:", len(model_features))
    print(
        "Single-window inference median/p95/p99 ms:",
        round(single["median_ms"], 4),
        round(single["p95_ms"], 4),
        round(single["p99_ms"], 4)
    )
    print(
        "Batch inference rows:",
        n_batch,
        "| median/p95/p99 ms:",
        round(batch["median_ms"], 4),
        round(batch["p95_ms"], 4),
        round(batch["p99_ms"], 4)
    )

else:
    print(
        "INFERENCE PROFILING INCOMPLETE: model or prepared input X "
        "is unavailable. Check Section 2."
    )

rss_after = rss_mb()
print(
    "Process RSS before (MB):", rss_before,
    "| after (MB):", rss_after
)


Selected artifact: outputs\notebook06_v2\models\nb6_W_F.joblib
Model type: RandomForestClassifier
Model load seconds: 3.258698
Input rows available: 11826
Features per row: 15
Single-window inference median/p95/p99 ms: 203.8425 302.7105 328.4242
Batch inference rows: 256 | median/p95/p99 ms: 290.7183 311.6966 327.2991
Process RSS before (MB): 125.1171875 | after (MB): 222.77734375


## 4A. End-to-end software replay (diagnostic only)
Run the saved estimator on a small slice, expose class probabilities and the NB6 decision thresholds. This is a replay diagnostic, not a physical sensor or correction-magnitude implementation.


In [11]:
# Small deterministic replay through saved preprocessing + estimator + saved thresholds.
# It deliberately emits a decision recommendation only; it does not mutate sensor values.
REPLAY_N = min(100, len(X))
replay_X = X.iloc[:REPLAY_N].copy()
replay_prob = predict_for_profile(model, replay_X)
replay_rows = pd.DataFrame({"row_index": replay_X.index})
if hasattr(model, "predict_proba"):
    classes = list(getattr(model, "classes_", []))
    if replay_prob.ndim != 2 or replay_prob.shape[1] != len(classes):
        raise ValueError("Estimator probability shape does not match its class labels.")
    replay_rows["predicted_class_raw"] = np.asarray(classes, dtype=object)[np.argmax(replay_prob, axis=1)]
    replay_rows["max_probability_raw"] = np.max(replay_prob, axis=1)
    for j, label in enumerate(classes):
        replay_rows[f"p_{label}"] = replay_prob[:, j]
    tau = artifact_metadata.get("tau")
    t_corr = artifact_metadata.get("t_corr")
    drift_idx = classes.index("DRIFT") if "DRIFT" in classes else None
    if tau is not None:
        replay_rows["confidence_gate_accept_raw"] = replay_rows["max_probability_raw"] >= float(tau)
    if t_corr is not None and drift_idx is not None:
        # Match NB6 decision semantics: correction attempt is based on predicted
        # DRIFT and P(DRIFT)>=t_corr; confidence tau does not gate this branch.
        pred_drift = replay_rows["predicted_class_raw"].eq("DRIFT")
        replay_rows["correction_attempt_raw"] = pred_drift & (replay_prob[:, drift_idx] >= float(t_corr))
    replay_rows.to_csv(OUT / "nb8_replay_diagnostics.csv", index=False)
    print("Replay rows:", len(replay_rows), "| saved: nb8_replay_diagnostics.csv")
    display(replay_rows.head(10))
else:
    print("Replay class-probability diagnostics skipped: estimator has decision_function only.")


Replay rows: 100 | saved: nb8_replay_diagnostics.csv


,row_index,predicted_class_raw,max_probability_raw,p_DRIFT,p_EVENT,p_EVENT_DRIFT,p_NORMAL,confidence_gate_accept_raw,correction_attempt_raw
0,0,NORMAL,0.734251,0.149147,0.026766,0.089836,0.734251,True,False
1,1,NORMAL,0.683789,0.161945,0.038468,0.115798,0.683789,True,False
2,2,NORMAL,0.671772,0.176350,0.045615,0.106263,0.671772,True,False
3,3,NORMAL,0.671081,0.168041,0.075602,0.085277,0.671081,True,False
4,4,NORMAL,0.490880,0.344871,0.057989,0.106260,0.490880,False,False
5,5,NORMAL,0.507720,0.396182,0.022054,0.074044,0.507720,False,False
6,6,DRIFT,0.637390,0.637390,0.030989,0.068380,0.263241,True,True
7,7,DRIFT,0.643774,0.643774,0.018097,0.047608,0.290522,True,True
8,8,DRIFT,0.676219,0.676219,0.021649,0.042110,0.260022,True,True
9,9,DRIFT,0.678653,0.678653,0.031309,0.042114,0.247923,True,True


## 5. Feature-table I/O benchmark (not online feature computation)

In [13]:
io_rows = []
if FEATURE_PATH.exists():
    io = benchmark(lambda: pd.read_parquet(FEATURE_PATH), warmup=1, repeats=5)
    io_rows.append({"measurement":"read_feature_parquet", "file_size_mb":FEATURE_PATH.stat().st_size/(1024**2), "rows":len(features) if features is not None else None, "columns":features.shape[1] if features is not None else None, **io})
    print("Parquet read median/p95 ms:", round(io["median_ms"],3), round(io["p95_ms"],3))
    print("This is batch file I/O only, not per-window online feature generation.")
else: print("Skipped: feature table not found.")

Parquet read median/p95 ms: 65.172 66.898
This is batch file I/O only, not per-window online feature generation.


## 6. Save results and document limitations

In [15]:
LATENCY = pd.DataFrame(latency_rows); IO_METRICS = pd.DataFrame(io_rows)
summary = {
 "timestamp_utc": pd.Timestamp.now(tz="UTC").isoformat(), "project_root": str(ROOT),
 "python_version": sys.version.split()[0], "platform": platform.platform(),
 "processor": platform.processor() or None, "logical_cpu_count": os.cpu_count(),
 "psutil_available": psutil is not None,
 "model_artifact": str(model_path.relative_to(ROOT)) if model_path else None,
 "model_type": type(model).__name__ if model is not None else None,
 "model_load_seconds": model_load_seconds,
 "feature_dataset": str(FEATURE_PATH.relative_to(ROOT)) if FEATURE_PATH.exists() else None,
 "feature_dataset_bytes": FEATURE_PATH.stat().st_size if FEATURE_PATH.exists() else None,
 "input_rows_available": int(len(X)) if X is not None else None,
 "inference_profiled": bool(inference_profiled and len(latency_rows) >= 2),
 "inference_measurements": [r.get("measurement") for r in latency_rows],
 "process_rss_before_mb": float(rss_before) if np.isfinite(rss_before) else None,
 "process_rss_after_mb": float(rss_after) if np.isfinite(rss_after) else None,
 "interpretation": "Measurements apply only to the machine running this notebook; desktop measurements are not gateway-device evidence."
}
display(LATENCY.round(4)); display(IO_METRICS.round(4))
display(pd.DataFrame([summary]).T.rename(columns={0:"value"}))
LATENCY.to_csv(OUT/"nb8_latency_metrics.csv", index=False)
IO_METRICS.to_csv(OUT/"nb8_io_metrics.csv", index=False)
(OUT/"nb8_runtime_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved outputs to:", OUT)
if summary["inference_profiled"]:
    print("Inference profiled on this host. For an edge claim, run on the target gateway and report its hardware/OS.")
else:
    print("Inference profiling INCOMPLETE: no compatible fitted estimator was available.")
    print("Save the fitted NB6 estimator (including preprocessing) under outputs/ or models/, then rerun Sections 3–6.")
print("Limitations: RSS is whole-process memory; full online feature-generation latency is not measured; no accuracy or safety claim is made.")

,measurement,rows_per_call,repeats,median_ms,p95_ms,p99_ms,mean_ms
0,single_window_inference,1,200,203.8425,302.7105,328.4242,216.4896
1,batch_inference,256,50,290.7183,311.6966,327.2991,291.5584


,measurement,file_size_mb,rows,columns,repeats,median_ms,p95_ms,p99_ms,mean_ms
0,read_feature_parquet,1.6172,11826,53,5,65.1717,66.8979,67.1812,65.369


,value
timestamp_utc,2026-10-20T05:28:04.426410+00:00
project_root,C:\AIoT_Research
python_version,3.12.7
platform,Windows-10-10.0.19045-SP0
processor,"Intel64 Family 6 Model 142 Stepping 9, Genuine..."
logical_cpu_count,4
psutil_available,True
model_artifact,outputs\notebook06_v2\models\nb6_W_F.joblib
model_type,RandomForestClassifier
model_load_seconds,3.258698


Saved outputs to: C:\AIoT_Research\outputs\notebook08
Inference profiled on this host. For an edge claim, run on the target gateway and report its hardware/OS.
Limitations: RSS is whole-process memory; full online feature-generation latency is not measured; no accuracy or safety claim is made.


In [23]:
from pathlib import Path
import pandas as pd

root = Path(r"C:\AIoT_Research")
src = root / "outputs" / "notebook06_v2" / "nb6_v2_predictions.parquet"
out = root / "outputs" / "notebook08" / "nb6_predictions_for_audit.csv"

df = pd.read_parquet(src)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)

print("Exported:", out)


Shape: (25536, 16)
Columns: ['window_id', 'episode_id', 'variant', 'label', 'split', 'regime', 'set', 'part', 'calibrator', 'p_NORMAL', 'p_DRIFT', 'p_EVENT', 'p_EVENT_DRIFT', 'pred', 'confidence', 'decision']
Exported: C:\AIoT_Research\outputs\notebook08\nb6_predictions_for_audit.csv
